# 🏥 CareFlow — Notebook 17: Cohort Benchmark & A/B Testing
## Side-by-Side Dimensional Process Mining, Welch's t-test & Clinical Stratification

**Author:** CareFlow Data Science Team  
**Dataset:** `careflow_analytics/output/fct_patient_journeys.parquet`

---

### Objectives:
1. Conduct A/B comparative testing between **Paperwork Loopback Cohort** vs **First-Pass Clean Cohort**.
2. Benchmark **Morning Rush (8-11 AM)** vs **Nominal Hours**.
3. Calculate statistical effect size (Cohen's $d$) and two-sample Welch's t-test $p$-values.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

fct_df = pd.read_parquet('../careflow_analytics/output/fct_patient_journeys.parquet')

# Cohort A: Loopback Rework
cohort_loopback = fct_df[fct_df['has_radiology_loopback_rework']]['total_los_min']
# Cohort B: First-Pass Clean
cohort_clean = fct_df[~fct_df['has_radiology_loopback_rework']]['total_los_min']

t_stat, p_val = stats.ttest_ind(cohort_loopback, cohort_clean, equal_var=False)

# Cohen's d effect size
n1, n2 = len(cohort_loopback), len(cohort_clean)
s1, s2 = np.var(cohort_loopback, ddof=1), np.var(cohort_clean, ddof=1)
s_pooled = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
cohens_d = (cohort_loopback.mean() - cohort_clean.mean()) / s_pooled

print(f"Cohort A/B Comparative Analysis (Radiology Loopback vs Clean):")
print(f"  Loopback Cohort Mean LOS:   {cohort_loopback.mean():.1f} min (n={n1})")
print(f"  Clean Cohort Mean LOS:      {cohort_clean.mean():.1f} min (n={n2})")
print(f"  Observed Dwell Delta:       +{cohort_loopback.mean() - cohort_clean.mean():.1f} minutes")
print(f"  Welch's t-statistic:        {t_stat:.4f}")
print(f"  p-value:                    {p_val:.5e}")
print(f"  Cohen's d Effect Size:      {cohens_d:.3f} ({'Large Effect' if abs(cohens_d) > 0.8 else 'Moderate Effect'})")
